# Lesson 04 - 工具使用设计模式

在本课中，你将学习使用 Microsoft Agent Framework（Python）为 AI 代理设计的 **工具使用** 设计模式。我们涵盖：

- 使用 `@tool` 装饰器和类型化参数定义功能工具
- 提供工具架构，让模型了解每个工具的功能
- 使用 `approval_mode` 控制工具执行
- 通过 Pydantic 模型和 `response_format` 返回 **结构化输出**

场景是一个能够查询目的地、检查可用性和检索航班信息的 **旅行预订代理**。


## 设置


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 装饰器定义工具

`@tool` 装饰器将普通的 Python 函数转换为代理可以调用的工具。  
关键点：

- **文档字符串** 会成为模型看到的工具描述。  
- **类型注解**（包括带描述的 `Annotated`）定义工具的模式。  
- `approval_mode` 控制是否需要用户在每次调用执行前进行批准。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 创建一个具有多个工具的代理

将所有三个工具传递给客户端，以便模型可以调用其中任何一个来回答用户的问题。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## 使用工具的结构化输出

通过将 `response_format` 设置为 Pydantic 模型，代理程序被强制返回一个类型良好的 JSON 对象，而不是自由格式的文本。当下游代码需要以编程方式使用结果时，这非常有用。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## 工具审批模式

`@tool` 上的 `approval_mode` 参数控制工具调用在执行前是否需要人工审批：

| 模式 | 行为 |
|---|---|
| `"never_require"` | 工具自动运行——无需用户确认。 |
| `"always_require"` | 每次调用都必须经过用户批准后才能执行。 |

对于具有副作用的工具（例如预订航班、信用卡扣费），请使用 `"always_require"`，以便人工介入。


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

在本课中，您学习了如何：

1. 使用带有类型参数和作为工具模式的文档字符串的 `@tool` 装饰器**定义工具**。
2. **组合多个工具**，使代理能够按顺序调用它们以回答复杂查询。
3. 通过传递 Pydantic 模型作为 `response_format` **返回结构化输出**。
4. 通过 `approval_mode` **控制工具审批**，在敏感操作时保持人工参与。

这些模式构成了构建可靠、可投入生产的代理以安全地与外部系统交互的基础。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件已使用人工智能翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意自动翻译可能存在错误或不准确之处。原始文本的母语版本应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或误释，我们不承担任何责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
